# Preprocessing Pipeline — SSGI 2024 Stunting Risk Factors

This notebook executes the full data integration and standardization workflow for Fuzzy C-Means (FCM) clustering.

**Inputs**
- `data/interim/family_stunting_indicators.csv`
- `data/raw/water_access.csv`
- `data/raw/sanitation_access.csv`

**Outputs**
- `data/processed/fcm_risk_profile_2024.csv` (unstandardized)
- `data/processed/fcm_model_matrix_zscore.csv` (Z-score standardized)
- `data/processed/correlation_matrix.png`

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import (
    OUTPUT_CORRELATION_PLOT,
    OUTPUT_RISK_PROFILE,
    OUTPUT_ZSCORE_MATRIX,
    export_csv,
    load_family_indicators,
    load_sanitation_access,
    load_water_access,
    merge_datasets,
    run_quality_checks,
    save_correlation_heatmap,
    standardize_fcm_features,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
print(f"Project root: {PROJECT_ROOT}")

## 1. Load & Harmonize Source Datasets

Province names are stripped, uppercased, and the national `INDONESIA` aggregate row is removed.

In [ ]:
family_df = load_family_indicators()
water_df = load_water_access()
sanitation_df = load_sanitation_access()

print(f"Family indicators: {len(family_df)} provinces")
print(f"Water access:      {len(water_df)} provinces")
print(f"Sanitation access: {len(sanitation_df)} provinces")

display(family_df.head(3))
display(water_df.head(3))
display(sanitation_df.head(3))

## 2. Merge & Transform

Derived features:
- `water_no_or_unimproved_pct` = `water_no_access_pct` + `water_unimproved_pct`
- `sanitation_babs_pct` = `sanitation_open_babs_pct` + `sanitation_closed_babs_pct`

In [ ]:
merged_df = merge_datasets(family_df, water_df, sanitation_df)
merged_df

## 3. Data Quality Checks

Missing values are reported (not imputed with zero). Outliers are flagged via IQR and Z-score methods.

In [ ]:
quality_report = run_quality_checks(merged_df)

print("Missing values:")
display(quality_report["missing"])

print("\nIQR outliers:")
display(quality_report["outliers_iqr"])

print("\nZ-score outliers:")
display(quality_report["outliers_zscore"])

## 4. Correlation Matrix Heatmap

In [ ]:
from IPython.display import Image, display

save_correlation_heatmap(merged_df, OUTPUT_CORRELATION_PLOT)
display(Image(filename=str(OUTPUT_CORRELATION_PLOT)))

## 5. Z-Score Standardization (6 FCM Features)

`stunting_prevalence_pct` is retained only in the unstandardized output for post-clustering validation.

In [ ]:
zscore_df = standardize_fcm_features(merged_df)
zscore_df.describe()

## 6. Export Processed Files

In [ ]:
export_csv(merged_df, OUTPUT_RISK_PROFILE)
export_csv(zscore_df, OUTPUT_ZSCORE_MATRIX)

print(f"Unstandardized: {OUTPUT_RISK_PROFILE} ({len(merged_df)} rows)")
print(f"Z-score matrix: {OUTPUT_ZSCORE_MATRIX} ({len(zscore_df)} rows)")
print(f"Correlation plot: {OUTPUT_CORRELATION_PLOT}")